<a href="https://colab.research.google.com/github/ericksbr/deduper/blob/main/deduper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About Deduper

Deduper is a simple tool that finds **duplicate and modified comments** in rulemaking submissions. It was designed to work with comments submitted to the Federal Register, but should work with comments in a spreadsheet format.

## What it does
It reads a file of comments (CSV, Excel, or Parquet), groups similar ones together, and labels each comment as:
- **Duplicate** — word-for-word or nearly identical (Jaccard >= 0.95)
- **Modified** — very similar but with small changes (0.50 < Jaccard < 0.95)
- **Unique** — distinct enough to be standalone (Jaccard <= 0.50)

## How to use
1. Collapse the "Setup" section. Click the play icon next to "1 cell hidden" to load the required packges.
2. Collapse the "Function Definitions" section. Click the play icon next to "18 cells hidden" to define all the actions this pipeline completes. Advanced users can review the functions.
3. Click play in the Upload Fil section. Select your comment file (CSV, Excel, or Parquet). Click play in the next cell to confirm the Column and ID (optional). If you need to change these, you can do so in the next cell.
4. Collapse "Pipeline Execution" and click play to run the duplicate detection pipeline. Once running, you can expand and scroll to see progress at each stage.
5. Click play under the Interactive Explorer to browse groups of similar documents.
6. Download your file with new columns added for duplicate filtering.

## Key concept
Jaccard similarity measures how many word segments two comments share (0 = different, 1 = identical).

# Setup

In [39]:
# Install dependencies
import subprocess
import sys

packages = ['pandas', 'numpy', 'scipy', 'scikit-learn', 'openpyxl', 'pyarrow', 'nltk', 'tqdm']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

# Imports
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from difflib import SequenceMatcher
import unicodedata
import re
import time
import gc
from ipywidgets import Dropdown, Output, IntText, VBox, HBox, HTML
from IPython.display import display, clear_output
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
import nltk
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

# Google Colab file upload
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"âœ“ Setup complete ({'Google Colab' if IN_COLAB else 'Local Jupyter'})")

âœ“ Setup complete (Google Colab)


# Function Definitions (By Stage)

## Utilities

In [23]:
def load_file(filepath):
    """Load CSV/XLSX/Parquet file as DataFrame."""
    ext = Path(filepath).suffix.lower()
    if ext == '.csv':
        return pd.read_csv(filepath, dtype=str, keep_default_na=False)
    elif ext in ['.xlsx', '.xls']:
        return pd.read_excel(filepath, dtype=str, keep_default_na=False)
    elif ext == '.parquet':
        return pd.read_parquet(filepath).astype(str)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

def detect_comment_column(df):
    """Auto-detect comment column (longest text on average)."""
    return max(df.columns, key=lambda c: df[c].astype(str).str.len().mean())

def detect_id_column(df):
    """Auto-detect ID column."""
    for col in ['id', 'ID', 'comment_id', 'document_id', 'tracking_number', 'Document ID']:
        if col in df.columns:
            return col
    return None

## Stage 1: Exact Match (Raw Text)

In [24]:
def exact_dedup_raw(texts):
    """Exact match on raw text."""
    df_raw = pd.DataFrame({'text': texts})
    exact_grp_raw = df_raw.groupby('text', sort=False).ngroup() + 1
    is_first_seen_raw = ~df_raw.duplicated('text', keep='first').values
    return exact_grp_raw.values, is_first_seen_raw

## Stage 2: Exact Match (Normalized Text)

In [25]:
def normalize_text(text: str) -> str:
    """Normalize text: NFKC, lowercase, NLTK tokenize, alphanumeric filter."""
    text = unicodedata.normalize('NFKC', text)
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if any(c.isalnum() for c in t)]
    text = ' '.join(tokens)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def exact_dedup_normalized(texts, is_first_seen_raw):
    """Exact match on normalized text (efficient: normalize first-seen-raw only)."""
    normalized_texts = np.empty(len(texts), dtype=object)
    raw_to_norm = {}

    # Normalize only first-seen-raw
    for idx in tqdm(np.where(is_first_seen_raw)[0], desc='Normalizing', unit='doc'):
        raw_text = texts[idx]
        norm_text = normalize_text(raw_text)
        normalized_texts[idx] = norm_text
        raw_to_norm[raw_text] = norm_text

    # Propagate to exact raw-text duplicates
    for idx in np.where(~is_first_seen_raw)[0]:
        raw_text = texts[idx]
        normalized_texts[idx] = raw_to_norm.get(raw_text, "")

    # Group on normalized text
    df_norm = pd.DataFrame({'text_normalized': normalized_texts})
    exact_grp_normalized = df_norm.groupby('text_normalized', sort=False).ngroup() + 1
    is_first_seen_normalized = ~df_norm.duplicated('text_normalized', keep='first').values

    return normalized_texts, exact_grp_normalized.values, is_first_seen_normalized

## Stage 3: Shingles & MinHash

In [26]:
def build_shingle_matrix(texts, k=5):
    """Build sparse shingle matrix (docs Ã— shingles)."""
    all_shingles_set = set()
    doc_shingles = []

    for text in texts:
        words = text.split()
        if len(words) >= k:
            shingles = {' '.join(words[i:i+k]) for i in range(len(words) - k + 1)}
        else:
            shingles = {text}
        doc_shingles.append(shingles)
        all_shingles_set.update(shingles)

    shingle_list = sorted(list(all_shingles_set))
    shingle_to_idx = {s: i for i, s in enumerate(shingle_list)}

    rows, cols, data = [], [], []
    for doc_idx, shingles in enumerate(doc_shingles):
        for shingle in shingles:
            rows.append(doc_idx)
            cols.append(shingle_to_idx[shingle])
            data.append(1)

    X = csr_matrix((data, (rows, cols)), shape=(len(doc_shingles), len(shingle_list)), dtype=np.uint8)
    return X

def minhash_signatures(X, num_perm=256, seed=42):
    """Vectorized MinHash over permutations (not shingles)."""
    n_docs = X.shape[0]
    assert np.all(np.diff(X.indptr) > 0), 'Every doc needs at least one shingle'

    rng = np.random.default_rng(seed)
    a = rng.integers(1, 2**63, size=num_perm, dtype=np.uint64) | np.uint64(1)
    b = rng.integers(0, 2**63, size=num_perm, dtype=np.uint64)

    ids = X.indices.astype(np.uint64)
    starts = X.indptr[:-1]
    sig = np.full((n_docs, num_perm), np.uint64(2**64 - 1), dtype=np.uint64)

    step = max(1, int(512e6 // (8 * max(1, len(ids)))))
    for p0 in tqdm(range(0, num_perm, step), desc='MinHash', unit='chunk'):
        p1 = min(num_perm, p0 + step)
        h = ids[:, None] * a[p0:p1] + b[p0:p1]
        h ^= h >> np.uint64(31)
        h *= np.uint64(0x9E3779B97F4A7C15)
        h ^= h >> np.uint64(29)
        sig[:, p0:p1] = np.minimum.reduceat(h, starts, axis=0)

    return sig

## Stage 4: LSH & Jaccard Verification

In [27]:
def choose_lsh_params(num_perm, min_threshold, target_recall):
    """Largest rows-per-band r (fewest junk candidates) whose S-curve still gives
    P(candidate | J = min_threshold) >= target_recall."""
    best = None
    for r in range(1, num_perm + 1):
        b = num_perm // r
        if 1 - (1 - min_threshold**r) ** b >= target_recall:
            best = (b, r)
    if best is None:
        raise ValueError('No (b, r) reaches the target recall; raise NUM_PERM or lower TARGET_RECALL')
    return best

def _pairs_for_groups(order, starts, sz, n):
    """All within-group pairs for many groups of identical size sz, vectorized, encoded lo*n+hi."""
    members = order[starts[:, None] + np.arange(sz)]          # (G, sz)
    i, j = np.triu_indices(sz, 1)
    p, q = members[:, i].ravel(), members[:, j].ravel()
    return np.minimum(p, q) * n + np.maximum(p, q)

def lsh_candidate_pairs(sig, b, r, max_bucket=500, big_degree=50, seed=42):
    """Band the signatures, bucket identical bands, and emit within-bucket pairs.
    Memory-safe: consolidates every 30M pairs."""
    n = sig.shape[0]
    rng = np.random.default_rng(seed + 1)
    mult = rng.integers(1, 2**63, size=r, dtype=np.uint64) | np.uint64(1)
    chunks, pending, n_sampled = [], 0, 0

    for k in tqdm(range(b), desc='LSH bands', unit='band'):
        key = (sig[:, k * r:(k + 1) * r] * mult).sum(axis=1, dtype=np.uint64)
        order = np.argsort(key, kind='stable').astype(np.int64)
        ks = key[order]
        starts = np.r_[0, np.flatnonzero(ks[1:] != ks[:-1]) + 1]
        sizes = np.diff(np.r_[starts, n])

        for sz in np.unique(sizes[sizes > 1]):
            grp_starts = starts[sizes == sz]
            if sz <= max_bucket:
                chunks.append(_pairs_for_groups(order, grp_starts, int(sz), n))
            else:
                for s0 in grp_starts:
                    n_sampled += 1
                    m = order[s0:s0 + sz]
                    p = np.repeat(m, big_degree)
                    q = m[rng.integers(0, sz, size=sz * big_degree)]
                    keep = p != q
                    p, q = p[keep], q[keep]
                    chunks.append(np.minimum(p, q) * n + np.maximum(p, q))
            pending += len(chunks[-1])

        if pending > 30_000_000:  # consolidate to bound memory
            chunks = [np.unique(np.concatenate(chunks))]
            pending = len(chunks[0])

    if not chunks:
        return np.empty(0, np.int32), np.empty(0, np.int32), n_sampled

    codes = np.unique(np.concatenate(chunks))
    return codes // n, codes % n, n_sampled

def exact_jaccard(X, a, b):
    """Compute exact Jaccard for candidate pairs."""
    sizes = np.diff(X.indptr).astype(np.int64)
    inter = np.asarray(X[a].multiply(X[b]).sum(axis=1)).ravel()
    jaccard = inter / (sizes[a] + sizes[b] - inter)
    return jaccard

def exact_jaccard_batched(X, a, b, min_t, chunk=50_000):
    """Compute exact Jaccard in batches to avoid RAM spike."""
    verified_edges = []

    for chunk_start in range(0, len(a), chunk):
        chunk_end = min(chunk_start + chunk, len(a))
        chunk_a = a[chunk_start:chunk_end]
        chunk_b = b[chunk_start:chunk_end]

        # Compute Jaccard for this chunk
        sizes = np.diff(X.indptr).astype(np.int64)
        inter = np.asarray(X[chunk_a].multiply(X[chunk_b]).sum(axis=1)).ravel()
        jaccard = inter / (sizes[chunk_a] + sizes[chunk_b] - inter)

        # Keep only those >= min_t
        keep = jaccard >= min_t - 1e-9
        verified_edges.append((chunk_a[keep], chunk_b[keep], jaccard[keep]))

    if verified_edges:
        e1 = np.concatenate([e[0] for e in verified_edges]).astype(np.int32)
        e2 = np.concatenate([e[1] for e in verified_edges]).astype(np.int32)
        ej = np.concatenate([e[2] for e in verified_edges]).astype(np.float32)
        return e1, e2, ej
    else:
        return np.array([], dtype=np.int32), np.array([], dtype=np.int32), np.array([], dtype=np.float32)

def lsh_candidate_pairs_chunked(sig, b, r, max_bucket=500, big_degree=50, seed=42):
    """LSH with memory-safe chunking of pending pairs."""
    n = sig.shape[0]
    rng = np.random.default_rng(seed + 1)
    mult = rng.integers(1, 2**63, size=r, dtype=np.uint64) | np.uint64(1)
    all_pairs = []  # List of pair lists to consolidate
    n_sampled = 0

    for band_idx in tqdm(range(b), desc='LSH bands', unit='band'):
        key = (sig[:, band_idx*r:(band_idx+1)*r] * mult).sum(axis=1, dtype=np.uint64)
        order = np.argsort(key, kind='stable')
        ks = key[order]
        starts = np.r_[0, np.flatnonzero(ks[1:] != ks[:-1]) + 1]
        sizes = np.diff(np.r_[starts, n])

        band_pairs = []
        for sz in np.unique(sizes[sizes > 1]):
            grp_starts = starts[sizes == sz]
            for s0 in grp_starts:
                group = order[s0:s0+sz]
                for i in range(len(group)):
                    for j in range(i+1, len(group)):
                        lo, hi = min(group[i], group[j]), max(group[i], group[j])
                        band_pairs.append(lo * n + hi)

        if band_pairs:
            all_pairs.append(np.array(band_pairs, dtype=np.int64))

        # Consolidate when getting large
        if sum(len(p) for p in all_pairs) > 30_000_000:
            combined = np.unique(np.concatenate(all_pairs))
            all_pairs = [combined]

    if not all_pairs:
        return np.array([], dtype=np.int32), np.array([], dtype=np.int32), n_sampled

    # Decode back to (a, b)
    codes = np.unique(np.concatenate(all_pairs))
    idx_a = (codes // n).astype(np.int32)
    idx_b = (codes % n).astype(np.int32)
    return idx_a, idx_b, n_sampled


## Stage 5: Connected Components

In [28]:
def compute_groups_at_threshold(edge_a, edge_b, edge_j, threshold, n_docs):
    """Build connected components at a threshold."""
    mask = edge_j >= threshold - 1e-9
    if not mask.any():
        return np.arange(1, n_docs + 1, dtype=np.int32)

    a, b = edge_a[mask], edge_b[mask]

    row = np.concatenate([a, b])
    col = np.concatenate([b, a])
    data = np.ones(len(row), dtype=np.uint8)
    graph = csr_matrix((data, (row, col)), shape=(n_docs, n_docs))

    n_comp, labels = connected_components(graph, directed=False, return_labels=True)

    sizes = np.bincount(labels)
    order = np.argsort(-sizes)
    rank = np.empty(n_comp, dtype=np.int32)
    for new_grp, old_comp in enumerate(order, 1):
        rank[old_comp] = new_grp

    return rank[labels]

## Stage 6: Results Summary

In [29]:
def assign_category(row, group_ids_50, group_ids_90):
    """Assign category (Duplicate/Modified/Unique)."""
    campaign_size = (group_ids_50 == row['campaign_id']).sum()
    variant_size = (group_ids_90 == row['variant_id']).sum()

    if variant_size > 1:
        return 'Duplicate (Variant)'
    elif campaign_size > 1:
        return 'Modified (Campaign)'
    else:
        return 'Unique'

## Stage 7: Campaign Hierarchy

In [30]:
def get_jaccard_similarity(text1, text2, k=5):
    """Jaccard similarity on k-word shingles (normalized text, matching Stage 3)."""
    norm1 = normalize_text(str(text1))
    norm2 = normalize_text(str(text2))

    words1 = norm1.split()
    words2 = norm2.split()

    shingles1 = set(tuple(words1[i:i+k]) for i in range(max(0, len(words1) - k + 1)))
    shingles2 = set(tuple(words2[i:i+k]) for i in range(max(0, len(words2) - k + 1)))

    if not shingles1 or not shingles2:
        return 1.0 if len(words1) == len(words2) else 0.0

    intersection = len(shingles1 & shingles2)
    union = len(shingles1 | shingles2)
    return intersection / union if union > 0 else 0.0

def build_campaign_hierarchies(normalized_texts, group_ids_50):
    """Build campaign hierarchy: templates and buckets."""
    campaign_hierarchies = {}

    for campaign_id in tqdm(np.unique(group_ids_50), desc='Building hierarchy'):
        campaign_mask = group_ids_50 == campaign_id
        campaign_indices = np.where(campaign_mask)[0]

        # Campaign template (most-sent normalized text)
        campaign_norms = normalized_texts[campaign_indices]
        norm_counts = Counter(campaign_norms)
        template_norm = norm_counts.most_common(1)[0][0]
        template_idx = campaign_indices[np.where(campaign_norms == template_norm)[0][0]]

        # Jaccard to template
        jaccard_to_template = np.array([
            get_jaccard_similarity(normalized_texts[template_idx], normalized_texts[i])
            for i in campaign_indices
        ])

        # Bucket by similarity
        buckets = {'near_exact': [], 'high_var': [], 'med_var': [], 'low_var': []}
        for idx, j_sim in zip(campaign_indices, jaccard_to_template):
            if j_sim > 0.95:
                buckets['near_exact'].append((idx, j_sim))
            elif j_sim > 0.80:
                buckets['high_var'].append((idx, j_sim))
            elif j_sim > 0.60:
                buckets['med_var'].append((idx, j_sim))
            else:
                buckets['low_var'].append((idx, j_sim))

        # Variant template per bucket
        templates = {}
        for bucket_name, bucket_docs in buckets.items():
            if bucket_docs:
                bucket_norms = [normalized_texts[idx] for idx, _ in bucket_docs]
                bucket_counts = Counter(bucket_norms)
                most_common_norm = bucket_counts.most_common(1)[0][0]
                template_in_bucket = next(idx for idx, _ in bucket_docs
                                           if normalized_texts[idx] == most_common_norm)
                templates[bucket_name] = template_in_bucket

        campaign_hierarchies[campaign_id] = {
            'template': template_idx,
            'buckets': buckets,
            'templates': templates
        }

    return campaign_hierarchies

def select_representatives(normalized_texts, group_ids):
    """Select representative (most-sent normalized text) per group."""
    reps = {}
    for group_id in np.unique(group_ids):
        mask = group_ids == group_id
        group_texts = normalized_texts[mask]
        most_common = Counter(group_texts).most_common(1)[0][0]
        # Find first index with that text
        rep_idx = np.where(mask)[0][np.where(group_texts == most_common)[0][0]]
        reps[group_id] = rep_idx
    return reps

# Assign category based on actual J
def assign_category_by_j(j_sim, group_size):
    """Assign category based on Jaccard similarity."""
    if group_size == 1:
        return 'Unique'
    elif j_sim >= 0.95:
        return 'Duplicate'
    elif j_sim > 0.50:
        return 'Modified'
    else:
        return 'Unique'

## Stage 8: Interactive Explorer

In [31]:
BUCKET_LABELS = {
    'near_exact': 'Near-Exact (J > 0.95)',
    'high_var': 'High Similarity (0.80 - 0.95)',
    'med_var': 'Medium Similarity (0.60 - 0.80)',
    'low_var': 'Low Similarity (J < 0.60)',
    'all': 'All Documents'
}

def get_bucket_label(bucket_name):
    """Get descriptive label for bucket."""
    return BUCKET_LABELS.get(bucket_name, bucket_name)

def compare_docs_side_by_side(doc1_text, doc2_text):
    """Word-level diff and Jaccard similarity."""
    words1 = str(doc1_text).split()
    words2 = str(doc2_text).split()

    matcher = SequenceMatcher(None, words1, words2)
    word_sim = matcher.ratio()
    jaccard_sim = get_jaccard_similarity(doc1_text, doc2_text, k=5)

    left_html = right_html = ''
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            left_html += ' '.join(words1[i1:i2]) + ' '
            right_html += ' '.join(words2[j1:j2]) + ' '
        elif tag == 'delete':
            left_html += f'<span style="background: #ffcccc; text-decoration: line-through;">{" ".join(words1[i1:i2])}</span> '
        elif tag == 'insert':
            right_html += f'<span style="background: #ccffcc;">{" ".join(words2[j1:j2])}</span> '
        elif tag == 'replace':
            left_html += f'<span style="background: #ffcccc;">{" ".join(words1[i1:i2])}</span> '
            right_html += f'<span style="background: #ccffcc;">{" ".join(words2[j1:j2])}</span> '

    return jaccard_sim, word_sim, left_html, right_html

def build_explorer(results, comment_col, id_col):
    """Simplified explorer: groups sorted by J_to_rep."""
    # Filter to multi-member groups
    multi_groups = results.groupby('group_id').filter(lambda x: len(x) > 1)

    if len(multi_groups) == 0:
        print("⊘ No groups with 2+ members to explore")
        return

    # Build group list (sorted by size, descending)
    group_ids = multi_groups['group_id'].unique()
    group_options = {
        f"Group {int(gid)} · {(multi_groups['group_id'] == gid).sum()} copies": gid
        for gid in sorted(group_ids, key=lambda g: (multi_groups['group_id'] == g).sum(), reverse=True)
    }

    group_dropdown = Dropdown(
        options=group_options,
        value=list(group_options.values())[0],
        description='Group:',
        layout={'width': '600px'}
    )

    doc_idx_1 = IntText(value=0, description='Doc 1:', min=0, max=1000, layout={'width': '250px'})
    doc_idx_2 = IntText(value=1, description='Doc 2:', min=0, max=1000, layout={'width': '250px'})

    output_area = Output()

    def update_group(change):
        """Update doc list when group changes (sorted by J desc)."""
        group_id = change['new']
        group_rows = multi_groups[multi_groups['group_id'] == group_id].sort_values('J_to_rep', ascending=False)

        max_idx = len(group_rows) - 1
        doc_idx_1.max = max_idx
        doc_idx_2.max = max_idx
        doc_idx_1.value = 0
        doc_idx_2.value = min(1, max_idx)

        explore(group_id, 0, 1, group_rows)

    def explore(group_id, d1_idx, d2_idx, group_rows):
        """Display two docs side-by-side."""
        with output_area:
            clear_output(wait=True)

            group_rows_sorted = group_rows.sort_values('J_to_rep', ascending=False)

            if len(group_rows_sorted) < 2:
                display(HTML("<p style='color: orange;'>⊘ Not enough documents to compare</p>"))
                return

            d1_idx = min(max(d1_idx, 0), len(group_rows_sorted) - 1)
            d2_idx = min(max(d2_idx, 0), len(group_rows_sorted) - 1)

            if d1_idx == d2_idx:
                d2_idx = (d1_idx + 1) % len(group_rows_sorted)

            row1 = group_rows_sorted.iloc[d1_idx]
            row2 = group_rows_sorted.iloc[d2_idx]

            jaccard_sim, word_sim, left_html, right_html = compare_docs_side_by_side(
                row1[comment_col],
                row2[comment_col]
            )

            id1 = row1[id_col] if id_col else f"Doc {d1_idx}"
            id2 = row2[id_col] if id_col else f"Doc {d2_idx}"

            html = f"""
            <div style='font-family: sans-serif; font-size: 13px;'>
            <h4>Group {int(group_id)} (sorted by J to representative)</h4>

            <p style='margin: 8px 0; color: #666;'>
                <b>Jaccard:</b> <span style='color: #2E86AB; font-weight: bold;'>{jaccard_sim:.1%}</span>·
                <b>Word-level:</b> <span style='color: #2E86AB; font-weight: bold;'>{word_sim:.1%}</span>
            </p>

            <p style='margin: 4px 0; color: #666; font-size: 12px;'>
                Doc {d1_idx + 1} of {len(group_rows_sorted)} vs Doc {d2_idx + 1} of {len(group_rows_sorted)}
            </p>

            <hr style='margin: 8px 0; border: none; border-top: 1px solid #ddd;'>

            <table style='width: 100%; border-collapse: collapse; font-size: 12px;'>
            <tr style='background: #f0f0f0;'>
                <th style='text-align: left; padding: 6px; border: 1px solid #ccc;'>{id1}</th>
                <th style='text-align: left; padding: 6px; border: 1px solid #ccc;'>{id2}</th>
            </tr>
            <tr>
                <td style='padding: 8px; border: 1px solid #ddd; background: #fafafa; word-wrap: break-word; line-height: 1.5; max-height: 300px; overflow-y: auto;'>{left_html}</td>
                <td style='padding: 8px; border: 1px solid #ddd; background: #fafafa; word-wrap: break-word; line-height: 1.5; max-height: 300px; overflow-y: auto;'>{right_html}</td>
            </tr>
            </table>
            </div>
            """
            display(HTML(html))

    def on_group_change(change):
        update_group(change)

    def on_doc_idx_change(change):
        group_id = group_dropdown.value
        group_rows = multi_groups[multi_groups['group_id'] == group_id]
        explore(group_id, doc_idx_1.value, doc_idx_2.value, group_rows)

    group_dropdown.observe(on_group_change, names='value')
    doc_idx_1.observe(on_doc_idx_change, names='value')
    doc_idx_2.observe(on_doc_idx_change, names='value')

    display(VBox([
        HTML("<h3>Interactive Explorer: Groups & Comparisons</h3>"),
        HTML(f"<p style='font-size: 0.9em; color: #666;'>{len(group_ids)} groups with 2+ documents</p>"),
        group_dropdown,
        HBox([doc_idx_1, doc_idx_2]),
    ]))
    display(output_area)

    # Initialize first group
    group_id = group_dropdown.value
    group_rows = multi_groups[multi_groups['group_id'] == group_id].sort_values('J_to_rep', ascending=False)
    explore(group_id, 0, 1, group_rows)

# Upload Your File
Choose the file with comments from your computer.

In [1]:
if IN_COLAB:
    print("Click 'Choose Files' to upload your CSV/XLSX/Parquet file:")
    uploaded = files.upload()
    filepath = list(uploaded.keys())[0]
else:
    filepath = input("Enter path to your file (CSV/XLSX/Parquet): ")

print(f"File: {filepath}")

df = load_file(filepath)
print(f"Loaded {len(df):,} rows; {len(df.columns)} columns")

NameError: name 'IN_COLAB' is not defined

## Confirm Columns
Check to make sure the column with comments was correctly identified. If you have a commenter identification row, check this as well.

In [73]:
comment_col = detect_comment_column(df)
id_col = detect_id_column(df)

print(f"Comment column: {comment_col}")
print(f"ID column: {id_col or '(row index)'}")

Comment column: Comment
ID column: Document ID


## Override columns (optional)
If you need to change the comment or ID column, do so here.

In [ ]:
comment_dropdown = Dropdown(
    options=df.columns.tolist(),
    value=comment_col,
    description='Comment Column:',
    layout={'width': '500px'}
)

id_dropdown = Dropdown(
    options=['(none)'] + df.columns.tolist(),
    value=id_col if id_col else '(none)',
    description='ID Column:',
    layout={'width': '500px'}
)

output = Output()

def on_comment_change(change):
    global comment_col
    comment_col = change['new']
    update_display()

def on_id_change(change):
    global id_col
    id_col = None if change['new'] == '(none)' else change['new']
    update_display()

def update_display():
    with output:
        output.clear_output()
        sample = df[comment_col].fillna("").astype(str).iloc[0]
        print(f"âœ“ Comment column: {comment_col}")
        print(f"âœ“ ID column: {id_col or '(row index)'}")
        print(f"\nSample ({len(sample)} chars):")
        print(f"{sample[:300]}...")

comment_dropdown.observe(on_comment_change, names='value')
id_dropdown.observe(on_id_change, names='value')

display(VBox([
    HTML("<h3>Confirm Column Selection</h3>"),
    HTML("<p>Update if auto-detection is wrong:</p>"),
    comment_dropdown,
    id_dropdown,
    output
]))

update_display()

# Pipeline Execution
Run the duplicate detector by clicking play. Look at the subsections to understand progress along the way.

## Prepare Data

In [74]:
texts = df[comment_col].fillna("").astype(str).values
n_docs = len(texts)

print(f"{n_docs:,} documents")
print(f"  - Min length: {min((len(t) for t in texts), default=0)} chars")
print(f"  - Max length: {max((len(t) for t in texts), default=0):,} chars")
print(f"  - Mean length: {np.mean([len(t) for t in texts]):.0f} chars")

t_start = time.time()

258,399 documents
  - Min length: 0 chars
  - Max length: 5,000 chars
  - Mean length: 1285 chars


## Exact Match (Raw Text)

Finds comments that are identical to each other.

In [75]:
exact_grp_raw, is_first_seen_raw = exact_dedup_raw(texts)
exact_raw_count = (~is_first_seen_raw).sum()

print(f"Exact duplicates (raw): {exact_raw_count:,} ({100*exact_raw_count/n_docs:.1f}%)")
print(f"  Unique: {exact_grp_raw.max():,}")

Exact duplicates (raw): 46,183 (17.9%)
  Unique: 212,216


## Exact Match (Normalized Text)

Find comments that say the same thing, even if they differ slightly in capitalization, punctuation, or spacing. For example, 'I oppose this!' and 'i OPPOSE this' would be treated as different above (raw text) but identical here.

In [76]:
normalized_texts, exact_grp_normalized, is_first_seen_normalized = exact_dedup_normalized(texts, is_first_seen_raw)
exact_norm_count = (~is_first_seen_normalized).sum()

print(f"\nExact duplicates (normalized): {exact_norm_count:,} ({100*exact_norm_count/n_docs:.1f}%)")
print(f"Unique: {exact_grp_normalized.max():,}")

Normalizing: 100%|██████████| 212216/212216 [05:45<00:00, 613.96doc/s]



Exact duplicates (normalized): 48,860 (18.9%)
Unique: 209,539


## Shingles & MinHash

Splits each comment into overlapping 5-word chunks (called shingles), then creates a quick, stable mathematical fingerprint (MinHash) from those chunks so it can rapidly find which comments are similar.

In [77]:
fs_pos = np.where(is_first_seen_normalized)[0]
fs_texts = normalized_texts[fs_pos]
n_fs = len(fs_pos)

print(f"First-seen (normalized): {n_fs:,}")

X = build_shingle_matrix(fs_texts)
print(f"Shingle matrix: {X.shape[0]} docs; {X.shape[1]:,} shingles")

sig = minhash_signatures(X, num_perm=256)
print(f"\nSignatures: {sig.shape}")

First-seen (normalized): 209,539
Shingle matrix: 209539 docs; 5,652,029 shingles


MinHash: 100%|██████████| 256/256 [01:48<00:00,  2.36chunk/s]


Signatures: (209539, 256)


## LSH & Jaccard Verification

Organizes the fingerprints from above into groups (LSH bands) so similar comments end up together, then verifies which comments in each group actually are similar by counting how many 5-word chunks they share, keeping only pairs that share at least 50%.

In [ ]:
b, r = choose_lsh_params(256, 0.50, 0.995)
print(f"LSH: {b} bands × {r} rows per band")

cand_a, cand_b, n_sampled = lsh_candidate_pairs(sig, b, r)
print(f"✓ Generated {len(cand_a):,} candidate pairs")

# Verify and cleanup
if len(cand_a) > 0:
    edge_a, edge_b, edge_j = exact_jaccard_batched(X, cand_a, cand_b, min_t=0.50, chunk=50_000)
    edge_a = fs_pos[edge_a].astype(np.int32)
    edge_b = fs_pos[edge_b].astype(np.int32)
else:
    edge_a = edge_b = edge_j = np.array([], dtype=np.int32)

# Explicit cleanup
del cand_a, cand_b, X, sig
gc.collect()

print(f"✓ Verified: {len(edge_a):,} edges with J >= 0.50")

LSH: 85 bands × 3 rows per band


LSH bands:  61%|██████    | 52/85 [05:41<06:37, 12.03s/band]

## Connected Components

Groups comments together based on the similarity connections found in the previous step. If comment A is similar to B and B is similar to C, they all end up in the same group even if A and C aren't directly similar to each other. It does this at two thresholds: J>= 0.90 for strict groups and J>= 0.50 for broader groups.

In [ ]:
group_ids = compute_groups_at_threshold(edge_a, edge_b, edge_j, 0.50, n_docs)

## Build Hierarchy

For each group from the previous step, finds the most-sent version of the comment (the template) and then organizes all the variations by how similar they are to that main version. This creates a hierarchy from near-exact copies down to heavily modified versions.

In [ ]:
reps = select_representatives(normalized_texts, group_ids)

# Compute J between each comment and its group's representative
J_to_rep = np.zeros(n_docs)
for i in range(n_docs):
    group_id = group_ids[i]
    rep_idx = reps[group_id]
    J_to_rep[i] = get_jaccard_similarity(normalized_texts[i], normalized_texts[rep_idx])

## Results Summary

In [ ]:
elapsed = time.time() - t_start

print(f"Processing Time: {elapsed:.1f}s")

results = df.copy()
results['group_id'] = group_ids
results['J_to_rep'] = J_to_rep
group_sizes = pd.Series(group_ids).value_counts()
results['group_size'] = results['group_id'].map(group_sizes)
results['category'] = results.apply(
    lambda row: assign_category_by_j(row['J_to_rep'], row['group_size']),
    axis=1
)

print(results['category'].value_counts())

# Interactive Explorer

Tool that lets you browse the groups of similar comments, select any two to compare side-by-side, and see their differences highlighted with a similarity score. Use to verify that the grouping makes sense before downloading the results.

In [71]:
build_explorer(results, comment_col, id_col)

Output()

# Export Results

In [49]:
original_name = Path(filepath).stem
output_csv = f"{original_name}_deduper.csv"

results.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")
print(f"  Rows: {len(results):,}")
print(f"  Columns: {len(results.columns)}")

if IN_COLAB:
    files.download(output_csv)

Saved: fws_graywolf_2011 (1)_deduper.csv
  Rows: 14,577
  Columns: 64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>